# ArSL MobileNetV2 — Kaggle Edition

Arabic Sign Language recognition | 32 classes | 190,000 images | MobileNetV2

**Pipeline:** tf.data + AUTOTUNE | Two-phase training | TTA inference


## 1. Setup & Imports


In [ ]:
# Imports
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from scipy import ndimage


In [ ]:
# ── Hyperparameters / Reproducibility ────────────────────────────────────
IMG_SIZE        = 224
BATCH_SIZE      = 64
INITIAL_EPOCHS  = 8
FINETUNE_EPOCHS = 10
SEED            = 42
VAL_SPLIT       = 0.2  # used when we must create a val split

tf.keras.utils.set_random_seed(SEED)

# ── Runtime paths / dataset placeholders ─────────────────────────────────
OUTPUT_DIR   = '/kaggle/working' if os.path.exists('/kaggle/input') else '.'
DATASET_MODE = None   # 'split' | 'flat' | 'csv_train'
DATA_ROOT    = None
TRAIN_DIR    = None
VAL_DIR      = None
TEST_DIR     = None
LABELS_CSV   = None  # used when DATASET_MODE == 'csv_train'

print(f'IMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE} | OUTPUT_DIR={OUTPUT_DIR}')


In [ ]:
# ── GPU / Performance setup ─────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Keep float32 for stability; switch to 'mixed_float16' only if you need speed and know how to handle it.
tf.keras.mixed_precision.set_global_policy('float32')

gpu_names = [g.name for g in gpus] if gpus else ['None - CPU']
print(f'TF {tf.__version__} | GPUs: {gpu_names}')

AUTOTUNE = tf.data.AUTOTUNE


## 2. Dataset Labels & Class Counts

Hardcoded from `Number_of_images_per_Letter.csv` (32 classes, 190,000 images).


In [ ]:
# 32 Arabic letter classes — fallback list (may be overridden by CSV labels later)
KNOWN_COUNTS = {
    'ain': 5448, 'al': 5250, 'aleff': 5897, 'bb': 5380,
    'dal': 5227, 'dha': 5995, 'dhad': 6326, 'fa': 6858,
    'gaaf': 5630, 'ghain': 5850, 'ha': 6698, 'haa': 7092,
    'jeem': 6456, 'kaaf': 6435, 'khaa': 6679, 'la': 5784,
    'laam': 5510, 'meem': 5275, 'nun': 5760, 'ra': 5612,
    'saad': 5723, 'seen': 5139, 'sheen': 5240, 'ta': 6166,
    'taa': 6081, 'thaa': 6083, 'thal': 6805, 'toot': 6651,
    'waw': 5977, 'ya': 6354, 'yaa': 5511, 'zay': 5108,
}

# Default class list (if CSV labels are detected later, this will be replaced).
class_names = sorted(KNOWN_COUNTS.keys())
NUM_CLASSES = len(class_names)

print(f'Default classes ({NUM_CLASSES}): {class_names}')
print('Note: if a labels CSV is detected, class_names will be taken from it.')


## 3. Path Detection

Auto-detects Kaggle dataset structure — works for:

- **Split mode**: dataset has `train/` and `val/` subfolders
- **Flat mode**: dataset has class folders directly (80/20 split applied automatically)
- **CSV-train mode**: dataset has **only `train/`** plus a **labels CSV** (paths + labels)


In [ ]:
def _walk_limited(root, max_depth=4):
    for r, d, f in os.walk(root):
        level = r.replace(root, '').count(os.sep)
        if level > max_depth:
            d[:] = []
            continue
        yield r, d, f


def _score_labels_csv(csv_path):
    name = os.path.basename(csv_path).lower()
    score = 0
    for token in ['label', 'labels', 'train', 'annotation', 'annotations']:
        if token in name:
            score += 2
    try:
        head = pd.read_csv(csv_path, nrows=5)
    except Exception:
        return -1
    cols = {c.lower(): c for c in head.columns}
    has_label = any(k in cols for k in ['label', 'class', 'category', 'y'])
    has_path = any(k in cols for k in ['path', 'filepath', 'file_path', 'image', 'image_id', 'filename', 'file'])
    if has_label:
        score += 5
    if has_path:
        score += 5
    return score if (has_label and has_path) else -1


def _find_best_labels_csv(root):
    candidates = []
    for r, _, files in _walk_limited(root, max_depth=5):
        for fn in files:
            if not fn.lower().endswith('.csv'):
                continue
            p = os.path.join(r, fn)
            s = _score_labels_csv(p)
            if s >= 0:
                candidates.append((s, p))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (x[0], -len(x[1])), reverse=True)
    return candidates[0][1]


def _find_train_folder(root):
    # Prefer an actual 'train' folder anywhere under /kaggle/input
    for r, dirs, _ in _walk_limited(root, max_depth=6):
        if 'train' in dirs:
            return os.path.join(r, 'train')
    return None


if os.path.exists('/kaggle/input'):
    PRINT_INPUT_TREE = False  # set True for debugging
    if PRINT_INPUT_TREE:
        print('=== /kaggle/input structure (up to depth 4) ===')
        for _r, _d, _f in _walk_limited('/kaggle/input', max_depth=4):
            _level = _r.replace('/kaggle/input', '').count(os.sep)
            _indent = '  ' * _level
            print(f'{_indent}{os.path.basename(_r)}/')
        print('=' * 50)

    # Try 1: find folder with pre-made train/ and val/ subdirectories
    for _root, _dirs, _ in _walk_limited('/kaggle/input', max_depth=6):
        if 'train' in _dirs and 'val' in _dirs:
            DATA_ROOT    = _root
            TRAIN_DIR    = os.path.join(_root, 'train')
            VAL_DIR      = os.path.join(_root, 'val')
            TEST_DIR     = os.path.join(_root, 'test')
            DATASET_MODE = 'split'
            print('Mode: SPLIT  (train/val folders exist)')
            print('TRAIN_DIR: ' + TRAIN_DIR)
            print('VAL_DIR  : ' + VAL_DIR)
            break

    # Try 2: train-only + labels CSV (common Kaggle structure)
    if DATASET_MODE is None:
        _train = _find_train_folder('/kaggle/input')
        _csv   = _find_best_labels_csv('/kaggle/input')
        if _train and _csv:
            TRAIN_DIR    = _train
            DATA_ROOT    = os.path.dirname(_train)
            LABELS_CSV   = _csv
            DATASET_MODE = 'csv_train'
            print('Mode: CSV_TRAIN  (train/ + labels CSV)')
            print('TRAIN_DIR : ' + TRAIN_DIR)
            print('LABELS_CSV: ' + LABELS_CSV)

    # Try 3: flat structure with class folders in root
    if DATASET_MODE is None:
        for _root, _dirs, _ in _walk_limited('/kaggle/input', max_depth=6):
            if len(set(_dirs) & set(class_names)) >= 20:
                DATA_ROOT    = _root
                DATASET_MODE = 'flat'
                print('Mode: FLAT  (class folders directly in root)')
                print('DATA_ROOT: ' + DATA_ROOT)
                break

    if DATASET_MODE is None:
        raise FileNotFoundError(
            'Cannot locate a compatible dataset under /kaggle/input.\n'
            'Expected one of:\n'
            '  - train/ + val/\n'
            '  - train/ + labels.csv\n'
            '  - flat class folders\n'
            'Attach the dataset to this notebook and re-run.'
        )

else:
    # Local Windows paths (kept as fallback)
    _BASE = (
        r'M:\Term 9\Grad\Main'
        r'\Sign-Language-Recognition-System-main'
        r'\Sign-Language-Recognition-System-main'
        r'\Sign_to_Sentence Project Main'
        r'\Datasets\Dataset (ArASL)\ArASL Database'
    )
    TRAIN_DIR    = os.path.join(_BASE, 'train')
    VAL_DIR      = os.path.join(_BASE, 'val')
    TEST_DIR     = os.path.join(_BASE, 'ArASL_35')
    DATA_ROOT    = _BASE
    DATASET_MODE = 'split' if os.path.exists(VAL_DIR) else 'csv_train'
    OUTPUT_DIR   = '.'

    if DATASET_MODE == 'csv_train':
        LABELS_CSV = _find_best_labels_csv(_BASE)
        print('Mode: CSV_TRAIN  (local)')
        print('TRAIN_DIR : ' + TRAIN_DIR)
        print('LABELS_CSV: ' + str(LABELS_CSV))
    else:
        print('Mode: SPLIT  (local)')
        print('TRAIN_DIR: ' + TRAIN_DIR)
        print('VAL_DIR  : ' + VAL_DIR)


In [ ]:
# If using train-only + labels CSV, load it and build a stratified train/val split.
csv_df = None
csv_train_paths = csv_train_labels = None
csv_val_paths   = csv_val_labels   = None

def _pick_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in cols_lower:
            return cols_lower[c]
    return None

def _resolve_paths(df, root_hint):
    path_col = _pick_col(df, ['path', 'filepath', 'file_path', 'image', 'filename', 'file'])
    if path_col is None:
        raise KeyError('Could not find an image path column in labels CSV.')
    raw = df[path_col].astype(str).tolist()

    resolved = []
    for p in raw:
        p2 = p.replace('\\', '/')
        if os.path.isabs(p2) and os.path.exists(p2):
            resolved.append(p2)
            continue
        # common: relative to dataset root, or relative to train folder
        cand1 = os.path.join(root_hint, p2)
        cand2 = os.path.join(TRAIN_DIR or root_hint, p2)
        if os.path.exists(cand1):
            resolved.append(cand1)
        elif os.path.exists(cand2):
            resolved.append(cand2)
        else:
            # last resort: join with basename under train/
            resolved.append(os.path.join(TRAIN_DIR or root_hint, os.path.basename(p2)))
    return resolved


if DATASET_MODE == 'csv_train':
    if LABELS_CSV is None:
        raise FileNotFoundError('DATASET_MODE=csv_train but LABELS_CSV was not found.')

    csv_df = pd.read_csv(LABELS_CSV)
    label_col = _pick_col(csv_df, ['label', 'class', 'category', 'y'])
    if label_col is None:
        raise KeyError('Could not find a label column in labels CSV.')

    csv_df = csv_df[[c for c in csv_df.columns if c is not None]].copy()
    csv_df[label_col] = csv_df[label_col].astype(str).str.strip()
    csv_df = csv_df[csv_df[label_col].notna() & (csv_df[label_col] != '')].copy()

    # Resolve paths and drop missing files
    csv_df['__path'] = _resolve_paths(csv_df, root_hint=DATA_ROOT or os.path.dirname(LABELS_CSV))
    csv_df = csv_df[csv_df['__path'].apply(lambda p: os.path.exists(p))].copy()
    if len(csv_df) == 0:
        raise FileNotFoundError('Labels CSV loaded, but no image files were found on disk after path resolution.')

    # Build class_names from CSV (overrides hardcoded list if different)
    csv_classes = sorted(csv_df[label_col].unique().tolist())
    if set(csv_classes) != set(class_names):
        class_names = csv_classes
        NUM_CLASSES = len(class_names)
        print(f'Class list taken from CSV. NUM_CLASSES={NUM_CLASSES}')
    else:
        print('CSV labels match the hardcoded class list.')

    label_to_index = {c: i for i, c in enumerate(class_names)}
    csv_df['__y'] = csv_df[label_col].map(label_to_index).astype(int)

    paths = csv_df['__path'].astype(str).tolist()
    y     = csv_df['__y'].astype(int).tolist()

    csv_train_paths, csv_val_paths, csv_train_labels, csv_val_labels = train_test_split(
        paths, y, test_size=VAL_SPLIT, random_state=SEED, stratify=y
    )
    print(f'CSV samples: total={len(paths):,} | train={len(csv_train_paths):,} | val={len(csv_val_paths):,}')


## 4. tf.data Pipeline


In [ ]:
# ── tf.data helpers ──────────────────────────────────────────────────────
SHUFFLE_BUFFER = 8192
IGNORE_BAD_IMAGES = True


def _with_fast_options(ds):
    options = tf.data.Options()
    # speed > determinism for training
    if hasattr(options, 'experimental_deterministic'):
        options.experimental_deterministic = False
    elif hasattr(options, 'deterministic'):
        options.deterministic = False
    return ds.with_options(options)


def _augment_fn(x):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.2)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return x


def make_dir_ds(directory, augment=False, subset=None, val_split=0.2):
    kwargs = dict(
        directory=directory,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        class_names=class_names,
        seed=SEED,
    )
    if subset is not None:
        kwargs['validation_split'] = val_split
        kwargs['subset'] = subset
    kwargs['shuffle'] = (subset == 'training') or (subset is None and augment)

    ds = tf.keras.utils.image_dataset_from_directory(**kwargs)

    def preprocess(x, y):
        x = tf.cast(x, tf.float32)
        x = preprocess_input(x)
        return x, y

    def augment_map(x, y):
        return _augment_fn(x), y

    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(augment_map, num_parallel_calls=AUTOTUNE)
    ds = _with_fast_options(ds)
    return ds.prefetch(AUTOTUNE)


def _decode_resize(path):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    img = tf.cast(img, tf.float32)
    return img


def make_csv_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED, reshuffle_each_iteration=True)

    def load_map(path, y):
        x = _decode_resize(path)
        x = preprocess_input(x)
        y = tf.one_hot(y, depth=NUM_CLASSES)
        return x, y

    ds = ds.map(load_map, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (_augment_fn(x), y), num_parallel_calls=AUTOTUNE)

    if IGNORE_BAD_IMAGES:
        ds = ds.apply(tf.data.experimental.ignore_errors())

    ds = ds.batch(BATCH_SIZE)
    ds = _with_fast_options(ds)
    return ds.prefetch(AUTOTUNE)


print('tf.data helpers ready.')


In [ ]:
# ── Build datasets ───────────────────────────────────────────────────────
if DATASET_MODE == 'split':
    train_ds = make_dir_ds(TRAIN_DIR, augment=True)
    val_ds   = make_dir_ds(VAL_DIR,   augment=False)
    print('Using pre-split train/val directories.')

elif DATASET_MODE == 'flat':
    train_ds = make_dir_ds(DATA_ROOT, augment=True,  subset='training',   val_split=VAL_SPLIT)
    val_ds   = make_dir_ds(DATA_ROOT, augment=False, subset='validation', val_split=VAL_SPLIT)
    print(f'Using flat structure with {int(VAL_SPLIT*100)}% validation split.')

elif DATASET_MODE == 'csv_train':
    if csv_train_paths is None or csv_val_paths is None:
        raise RuntimeError('csv_train mode selected but CSV split was not created. Run the CSV loading cell first.')
    train_ds = make_csv_ds(csv_train_paths, csv_train_labels, training=True)
    val_ds   = make_csv_ds(csv_val_paths,   csv_val_labels,   training=False)
    print('Using train-only directory + labels CSV (stratified split).')

else:
    raise ValueError(f'Unknown DATASET_MODE: {DATASET_MODE}')

print('Datasets ready.')


## 5. Model Architecture


In [ ]:
# ── Base model ───────────────────────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
 )
base_model.trainable = False

# ── Classification head ──────────────────────────────────────────────────
inputs  = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(512, activation='relu')(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)
print('Model graph built.')


In [ ]:
# ── Compile & summary ────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
 )
model.summary()


## 6. Phase 1 — Initial Training


In [ ]:
# ── Callbacks (Phase 1) ──────────────────────────────────────────────────
cb1 = [
    callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'mobilenet_arabic_best_initial.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
    callbacks.CSVLogger(
        os.path.join(OUTPUT_DIR, 'training_initial.csv')),
 ]
print('Callbacks ready (Phase 1).')


In [ ]:
print('=' * 60)
print('PHASE 1: INITIAL TRAINING')
print('=' * 60)

history1 = model.fit(
    train_ds,
    epochs=INITIAL_EPOCHS,
    validation_data=val_ds,
    callbacks=cb1,
 )


## 7. Phase 2 — Fine-Tuning


In [ ]:
# ── Unfreeze & compile (Phase 2) ─────────────────────────────────────────
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
 )
print('Fine-tune compile done.')


In [ ]:
# ── Callbacks (Phase 2) ──────────────────────────────────────────────────
cb2 = [
    callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'mobilenet_arabic_best_finetuned.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
    callbacks.CSVLogger(
        os.path.join(OUTPUT_DIR, 'training_finetune.csv')),
 ]
print('Callbacks ready (Phase 2).')


In [ ]:
print('=' * 60)
print('PHASE 2: FINE-TUNING')
print('=' * 60)

history2 = model.fit(
    train_ds,
    epochs=FINETUNE_EPOCHS,
    validation_data=val_ds,
    callbacks=cb2,
 )

model.save(os.path.join(OUTPUT_DIR, 'mobilenet_arabic_final.h5'))
print('Final model saved.')


## 8. Training History


In [ ]:
from pathlib import Path

initial_csv  = Path(OUTPUT_DIR) / 'training_initial.csv'
finetune_csv = Path(OUTPUT_DIR) / 'training_finetune.csv'

phase_dfs = []
split_epoch = None

if initial_csv.exists():
    df1 = pd.read_csv(initial_csv)
    df1['phase'] = 'initial'
    phase_dfs.append(df1)
    split_epoch = len(df1)
else:
    print(f'Note: missing {initial_csv}. Run Phase 1 training first.')

if finetune_csv.exists():
    df2 = pd.read_csv(finetune_csv)
    df2['phase'] = 'finetune'
    phase_dfs.append(df2)
else:
    print(f'Note: missing {finetune_csv}. Run Phase 2 training first.')

if not phase_dfs:
    print('No training CSV logs found yet. Run training cells, then re-run this cell.')
else:
    df = pd.concat(phase_dfs, ignore_index=True)

    # Create a continuous epoch index across phases (CSVLogger restarts epoch at 0 per fit())
    epoch_idx = []
    offset = 0
    for phase_df in phase_dfs:
        n = len(phase_df)
        epoch_idx.extend(list(range(offset + 1, offset + n + 1)))
        offset += n
    df = df.copy()
    df['epoch_idx'] = epoch_idx

    def _best_col(name):
        return name if name in df.columns else None

    acc_col = _best_col('accuracy') or _best_col('categorical_accuracy')
    val_acc_col = _best_col('val_accuracy') or _best_col('val_categorical_accuracy')
    loss_col = _best_col('loss')
    val_loss_col = _best_col('val_loss')

    missing_cols = [c for c in [acc_col, val_acc_col, loss_col, val_loss_col] if c is None]
    if missing_cols:
        print('CSV columns not as expected. Available columns:')
        print(', '.join(df.columns))
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(df['epoch_idx'], df[acc_col], 'o-', color='orange', label='Train Acc')
        ax1.plot(df['epoch_idx'], df[val_acc_col], 'o-', color='red', label='Val Acc')
        if split_epoch is not None and split_epoch > 0 and finetune_csv.exists():
            ax1.axvline(x=split_epoch + 0.5, color='green', linestyle='--', label='Fine-tune')
        ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2.plot(df['epoch_idx'], df[loss_col], 'o-', color='blue', label='Train Loss')
        ax2.plot(df['epoch_idx'], df[val_loss_col], 'o-', color='purple', label='Val Loss')
        if split_epoch is not None and split_epoch > 0 and finetune_csv.exists():
            ax2.axvline(x=split_epoch + 0.5, color='green', linestyle='--', label='Fine-tune')
        ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(Path(OUTPUT_DIR) / 'training_history.png', dpi=150)
        plt.show()

        best_val_acc = float(df[val_acc_col].max()) if len(df) else float('nan')
        print(f'Best Val Accuracy (from CSV): {best_val_acc:.4f}')


## 9. TTA Evaluation on Test Set


In [ ]:
def tta_augment(image):
    return np.array([
        image,
        ndimage.rotate(image,  5, reshape=False, mode='nearest'),
        ndimage.rotate(image, -5, reshape=False, mode='nearest'),
        np.clip(image * 1.1, 0, 255),
        np.clip(image * 0.9, 0, 255),
    ])

best_path = os.path.join(OUTPUT_DIR, 'mobilenet_arabic_best_finetuned.h5')
best_model = tf.keras.models.load_model(best_path) if os.path.exists(best_path) else model
print('Model loaded for inference.')

# Outputs used by the confusion-matrix + summary cells
pred_classes = None
pred_confs   = None
image_names  = None
accuracy     = 0.0

if DATASET_MODE == 'csv_train':
    # Evaluate on the CSV-derived validation split (no folder-per-class structure).
    if csv_val_paths is None or csv_val_labels is None:
        raise RuntimeError('csv_train mode: missing csv_val_paths/csv_val_labels. Run the CSV loading cell first.')

    EVAL_MAX = 2000  # keep evaluation fast; increase if you want full-val evaluation
    n = min(len(csv_val_paths), EVAL_MAX)
    paths = csv_val_paths[:n]
    y_true = np.array(csv_val_labels[:n], dtype=np.int32)

    ds_eval = make_csv_ds(paths, y_true.tolist(), training=False)
    probs = best_model.predict(ds_eval, verbose=0)
    pred_classes = np.argmax(probs, axis=1)
    pred_confs   = np.max(probs, axis=1)
    image_names  = [class_names[i] for i in y_true.tolist()]

    accuracy = float((pred_classes == y_true).mean() * 100.0)
    avg_conf = float(pred_confs.mean() * 100.0) if len(pred_confs) else 0.0
    print(f'Val samples evaluated: {n}/{len(csv_val_paths)}')
    print(f'Val Accuracy: {accuracy:.2f}%')
    print(f'Avg Confidence: {avg_conf:.1f}%')

else:
    # Folder-based evaluation (uses TEST_DIR if present, else VAL_DIR, else DATA_ROOT)
    _test_root = TEST_DIR if (TEST_DIR and os.path.exists(TEST_DIR)) else (
        VAL_DIR if (VAL_DIR and os.path.exists(VAL_DIR)) else DATA_ROOT
    )
    print('Test root: ' + str(_test_root))

    test_images, image_names = [], []
    for cls in sorted(os.listdir(_test_root)):
        cls_path = os.path.join(_test_root, cls)
        if not os.path.isdir(cls_path) or cls not in class_names:
            continue
        imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            img = cv2.imread(os.path.join(cls_path, imgs[0]))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
                test_images.append(img)
                image_names.append(cls)

    test_images = np.array(test_images)
    all_preds = []
    for img in test_images:
        t = preprocess_input(tta_augment(img).copy())
        p = best_model.predict(t, verbose=0, batch_size=5)
        all_preds.append(np.mean(p, axis=0))

    all_preds    = np.array(all_preds)
    pred_classes = np.argmax(all_preds, axis=1)
    pred_confs   = np.max(all_preds,   axis=1)

    correct  = sum(image_names[i] == class_names[pred_classes[i]] for i in range(len(image_names)))
    accuracy = correct / max(len(image_names), 1) * 100
    avg_conf = pred_confs.mean() * 100 if len(pred_confs) else 0

    print(f'Test images: {len(test_images)}')
    print(f'Test Accuracy (TTA): {accuracy:.2f}%')
    print(f'Avg Confidence: {avg_conf:.1f}%')
    print()
    for i, name in enumerate(image_names):
        pred = class_names[pred_classes[i]]
        conf = pred_confs[i] * 100
        mark = 'OK' if pred == name else 'XX'
        print(f'  [{mark}] {name:10s} -> {pred:10s} ({conf:.1f}%)')


## 10. Confusion Matrix


In [ ]:
pred_labels = [class_names[pred_classes[i]] for i in range(len(image_names))]
label_set   = sorted(set(class_names))
cm = confusion_matrix(list(image_names), pred_labels, labels=label_set)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_set, yticklabels=label_set)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — Test Set (TTA)', fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()


## 11. Final Summary


In [ ]:
print('=' * 60)
print('ARSL MOBILENETV2 KAGGLE EDITION - SUMMARY')
print('=' * 60)
print(f'  Mode:            {DATASET_MODE}')
print(f'  Classes:         {NUM_CLASSES}')
print(f'  Image size:      {IMG_SIZE}x{IMG_SIZE}')
print(f'  Batch size:      {BATCH_SIZE}')
print(f'  Initial epochs:  {INITIAL_EPOCHS}')
print(f'  Finetune epochs: {FINETUNE_EPOCHS}')
print(f'  Test Accuracy:   {accuracy:.2f}%')
print()
for fname in [
    'mobilenet_arabic_best_initial.h5',
    'mobilenet_arabic_best_finetuned.h5',
    'mobilenet_arabic_final.h5',
    'training_initial.csv',
    'training_finetune.csv',
    'training_history.png',
    'confusion_matrix.png',
]:
    path = os.path.join(OUTPUT_DIR, fname)
    mark = 'OK' if os.path.exists(path) else '--'
    print(f'  [{mark}] {fname}')
